# 추출된 이름을 판별하고 표준 ID에 연결하는 과제 LV1

rugplot과 jointplot의 실제 문서에서 추출한 8개 트리플을 읽고, 표기가 달라도 같은 개체로 판정한 출현을 묶고 그룹마다 ID를 부여합니다.  
교안 01과 교안 02를 모두 배운 뒤 수행하는 기초 복습 과제입니다.  
교안 01의 비교 후보 검색과 원문 판정, 교안 02의 그룹별 ID 부여를 적용합니다.  
1~10번은 순서대로 풀어 앞 문항의 결과를 이어 사용하고, 마지막에 과제 전용 JSONL 파일로 저장합니다.  
11번은 DB 없이 GDS 후보 검색의 입력과 유사도, 판정의 한계를 확인하는 필수 문항입니다.  
12번은 실제 GDS로 비교 후보 쌍을 찾는 선택 실습이며, 직전 준비 셀에서 Neo4j에 연결합니다.  
1~11번에는 DB 연결이나 API 키가 필요하지 않습니다.  


In [ ]:
# [제공코드] JSONL 자료를 읽고 이름과 출현 기록을 비교할 도구를 준비합니다.
import json
from pathlib import Path
from itertools import combinations
from difflib import SequenceMatcher
from pprint import pprint

data_dir = Path("data")

def load_rows(filename):
    """한 줄에 한 기록이 저장된 JSONL 파일을 딕셔너리 목록으로 읽습니다."""
    lines = (data_dir / filename).read_text(encoding="utf-8").splitlines()
    return [json.loads(line) for line in lines if line.strip()]

def pair_key(left_id, right_id):
    """비교 순서가 바뀌어도 같은 두 기록을 같은 키로 나타냅니다."""
    return tuple(sorted((left_id, right_id)))


## 저장된 추출 결과를 살펴봅니다

이 자료는 실제 Seaborn 문서에서 LLM이 추출해 저장한 트리플의 일부입니다.  
원문에서 트리플을 다시 추출하지 않고, 저장된 주어와 목적어를 정리합니다.  
**출현 기록**은 트리플 한 행의 주어 또는 목적어 자리를 따로 기록한 것입니다.  
`triple_id`는 추출 행의 ID, `mention_id`는 그 행의 어느 자리인지 구분하는 출현 ID입니다.  

| 출처 키 | 뜻 |
|---|---|
| source_file | 원본 추출 JSONL 파일 경로 |
| source_line | 그 파일의 행 번호. 1부터 시작 |
| source_triple_index | 해당 행 안의 트리플 번호. 1부터 시작 |

`evidence`와 `source_doc_id`는 원문으로 돌아가 판정 근거를 확인할 때 사용합니다.  


In [ ]:
# [제공코드] kg_lv1_triples.jsonl: 원문과 추출 위치가 보존된 과제용 트리플입니다.
task_triples = load_rows("kg_lv1_triples.jsonl")
triple_by_id = {row["triple_id"]: row for row in task_triples}
original_triples = [dict(row) for row in task_triples]

# kg_corpus.jsonl: 추출의 출처인 실제 문서의 본문과 URL입니다.
task_documents = {row["doc_id"]: row for row in load_rows("kg_corpus.jsonl")}
first_document = task_documents[task_triples[0]["source_doc_id"]]
print("첫 원문:", first_document["title"], first_document["url"])
print(first_document["text"][:500])

print("추출 트리플:", len(task_triples))
for row in task_triples:
    print(row["triple_id"], row["subject"], row["relation"], row["object"])
pprint(task_triples[0])


In [ ]:
# [제공코드] 근거 문장이 원문에 있는 경우와 없는 경우를 함께 담은 검사 입력입니다.
# 저장 자료와 분리해 두었으므로 실제 추출 결과가 아닙니다.
evidence_probe_documents = {
    "probe:doc1": {"doc_id": "probe:doc1", "text": "rugplot은 관측값의 위치를 축 위에 짧은 선으로 표시합니다."},
    "probe:doc2": {"doc_id": "probe:doc2", "text": "jointplot은 두 변수의 관계와 각 변수의 분포를 함께 그립니다."},
}
evidence_probe_triples = [
    {"triple_id": "p01", "source_doc_id": "probe:doc1", "evidence": "관측값의 위치를 축 위에 짧은 선으로 표시합니다"},
    {"triple_id": "p02", "source_doc_id": "probe:doc1", "evidence": "관측값의 개수를 막대 높이로 표시합니다"},
    {"triple_id": "p03", "source_doc_id": "probe:doc2", "evidence": "두 변수의 관계와 각 변수의 분포를 함께 그립니다"},
    {"triple_id": "p04", "source_doc_id": "probe:doc2", "evidence": "세 변수의 상관계수를 표로 정리합니다"},
]
pprint(evidence_probe_triples)


## 1. 추출 결과의 출처를 보존합니다

**배경**: 표준 ID를 연결한 뒤에도 어느 원문과 추출 행에서 왔는지 확인할 수 있어야 합니다. 원래 이름은 덮어쓰지 않습니다.  

**요구사항**  
- **provenance_rows** 에 각 task_triples 행의 `triple_id`, `source_doc_id`, `evidence`, `source_file`, `source_line`, `source_triple_index`를 복사한 딕셔너리 목록을 담으세요. 입력 순서를 유지하세요.
- **`evidence_in_document(triple, documents)`** 함수를 작성하세요. `triple`은 트리플 한 행, `documents`는 문서 ID를 문서에 연결한 딕셔너리입니다. triple의 evidence가 documents에서 source_doc_id로 찾은 문서의 text에 그대로 포함되면 True, 아니면 False를 반환합니다.
- **evidence_checks** 에 task_triples와 task_documents를 evidence_in_document로 검사한 `triple_id: bool`을 담으세요.
- **evidence_probe_checks** 에 같은 함수로 evidence_probe_triples와 evidence_probe_documents를 검사한 `triple_id: bool`을 담으세요.
- **provenance_rows** 의 첫 행과 두 검사 결과를 출력하고, task_triples의 첫 행에 있는 주어·관계·목적어와 함께 읽으세요.

**확인 기준**: 행 수는 원래 트리플 수와 같습니다. 검사 입력 4행 중 근거가 원문에 없는 행이 있어 evidence_probe_checks에는 False가 섞입니다. 원문 근거와 추출 위치는 원래 값을 그대로 보존합니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 원래 행에서 요구한 여섯 필드만 골라 새 딕셔너리에 담고, 포함 여부 검사는 두 입력에 같이 쓸 수 있도록 함수로 분리합니다.

세부구현:
1. 복사할 필드 이름을 목록으로 정합니다.
2. 트리플마다 필드의 값을 복사해 결과 목록에 추가합니다.
3. 트리플과 문서 딕셔너리를 받아 포함 여부를 돌려주는 함수를 만듭니다.
4. 저장 자료와 검사 입력에 같은 함수를 적용합니다.
5. 원래 트리플을 바꾸지 않았는지 확인합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
expected_keys = ["triple_id", "source_doc_id", "evidence", "source_file", "source_line", "source_triple_index"]
expected_provenance = []
for row in original_triples:
    expected_provenance.append({key: row[key] for key in expected_keys})
assert provenance_rows == expected_provenance, "원래 순서와 여섯 필드의 값을 보존하세요."
assert evidence_checks == {row["triple_id"]: row["evidence"] in task_documents[row["source_doc_id"]]["text"] for row in original_triples}, "각 evidence를 해당 source_doc_id의 문서에서 확인하세요."
# 저장 자료는 모두 포함이라 결과를 True로 가정해도 통과합니다. 검사 입력으로 실제 대조 여부를 확인합니다.
assert evidence_probe_checks == {"p01": True, "p02": False, "p03": True, "p04": False}, "검사 입력은 문서 text에서 직접 대조해야 True와 False가 갈립니다."
assert evidence_in_document(evidence_probe_triples[1], evidence_probe_documents) is False, "포함되지 않은 근거는 False를 돌려주세요."
assert all(isinstance(value, bool) for value in evidence_checks.values()), "포함 여부는 bool로 담으세요."
assert task_triples == original_triples, "입력 트리플을 직접 수정하지 마세요."
print("✅ 통과!")


## 2. 트리플의 양 끝을 출현 기록으로 만듭니다

**배경**: 같은 표기가 반복되어도 어느 트리플의 주어 또는 목적어인지 각각 남겨야 합니다.  

**요구사항**  
- **mentions** 를 리스트로 만들고, task_triples의 입력 순서대로 각 트리플의 주어, 목적어 출현 기록을 추가하세요.
- **mentions** 의 각 기록은 `mention_id`, `triple_id`, `role`, `name`, `entity_type`, `source_doc_id`, `evidence`의 7개 키를 가집니다. `mention_id`는 triple_id 뒤에 `:subject` 또는 `:object`를 붙이고, `role`은 `subject` 또는 `object`입니다.
- **mentions** 의 `name`은 원본의 subject 또는 object, `entity_type`은 각각 subject_type 또는 object_type에서 복사합니다. 나머지 필드는 원래 트리플에서 복사합니다.

**확인 기준**: 8개 트리플에서 16개 출현 기록이 나옵니다. 같은 함수가 세 번 등장하면 출현 기록도 세 개입니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 바깥 반복문은 트리플, 안쪽 반복문은 subject와 object를 순회합니다.

세부구현:
1. role에 따라 이름과 타입 필드를 선택합니다.
2. triple_id와 role을 조합해 중복 없는 출현 ID를 만듭니다.
3. 원문과 문서 ID를 함께 복사합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
expected_mentions = []
for triple in task_triples:
    for role in ["subject", "object"]:
        expected_mentions.append({
            "mention_id": triple["triple_id"] + ":" + role,
            "triple_id": triple["triple_id"], "role": role,
            "name": triple[role], "entity_type": triple[role + "_type"],
            "source_doc_id": triple["source_doc_id"], "evidence": triple["evidence"],
        })
assert mentions == expected_mentions, "각 트리플에서 주어, 목적어 순서로 7개 필드를 그대로 옮기세요."
assert len({row["mention_id"] for row in mentions}) == 2 * len(task_triples), "이름이 같아도 출현 ID를 합치지 마세요."
assert task_triples == original_triples, "원래 트리플은 수정하지 마세요."
print("✅ 통과!")


In [ ]:
# [제공코드] 앞뒤 공백만 제거하고 내부 공백과 원래 철자를 보존하는지 확인할 입력입니다.
normalization_inputs = [
    " kdeplot ",
    "FacetGrid",
    " sns.kdeplot ",
    " 파이썬 입문(개정판) ",
    "  파이썬  입문  ",
    "   ",
]
pprint(normalization_inputs)


## 3. 원래 이름과 비교용 이름을 구분합니다

**배경**: 실제 추출 이름에 정리할 공백이 없더라도 다른 입력에서 표기 정보가 손실되지 않는지 확인해야 합니다.  

**요구사항**  
- **`normalize`** 함수를 직접 작성하세요. 문자열 `name` 하나를 받아 이름 앞뒤의 공백만 제거한 문자열을 반환합니다. 중간 공백·대소문자·점으로 구분한 접두어·괄호는 그대로 보존하며, 공백만 있는 입력은 빈 문자열이 됩니다.
- **`normalization_results`** 에 normalization_inputs의 각 이름을 normalize로 처리한 문자열을 입력 순서대로 담으세요. 이 검사 입력은 실제 문서의 추출 결과가 아닙니다.
- **`prepared_mentions`** 에 mentions의 각 기록을 복사하고 `comparison_name`을 추가한 딕셔너리 목록을 담으세요. comparison_name은 해당 기록의 name을 normalize에 전달한 결과이며, 입력 순서를 유지합니다. mentions와 normalization_inputs는 수정하지 않습니다.

**확인 기준**: normalization_results는 문자열 6개의 리스트입니다. prepared_mentions는 원래 7개 필드를 보존하고 comparison_name만 추가한 16행입니다. 비교용 이름이 같아도 아직 동일 개체로 확정하지 않습니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 문자열 양 끝을 정리하는 연산과 문자열 내부를 바꾸는 연산을 구별합니다.

세부구현:
1. 앞뒤 공백만 정리하는 함수를 작성합니다.
2. 독립 검사 입력의 변환 결과를 차례로 저장합니다.
3. 출현 기록을 복사해 비교용 이름을 추가하고 원래 name을 보존합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
expected_inputs = [" kdeplot ", "FacetGrid",
                   " sns.kdeplot ", " 파이썬 입문(개정판) ",
                   "  파이썬  입문  ", "   "]
assert normalization_inputs == expected_inputs, "독립 검사 입력을 변경하지 마세요."
# 답안에서 바꿀 수 있는 mentions 자체를 기대값으로 사용하지 않습니다.
expected_original_mentions = []
for triple in original_triples:
    for role in ["subject", "object"]:
        expected_original_mentions.append({
            "mention_id": triple["triple_id"] + ":" + role,
            "triple_id": triple["triple_id"], "role": role,
            "name": triple[role], "entity_type": triple[role + "_type"],
            "source_doc_id": triple["source_doc_id"], "evidence": triple["evidence"],
        })
assert mentions == expected_original_mentions, "원래 출현의 모든 필드와 순서를 보존하세요."
assert normalization_results == [name.strip() for name in expected_inputs], "앞뒤 공백만 정리하고 입력 순서를 유지하세요."
for name in expected_inputs:
    assert normalize(name) == name.strip(), "앞뒤 공백만 제거하고 내부 철자와 공백을 보존하세요."
assert len(prepared_mentions) == len(mentions), "모든 출현 기록을 보존하세요."
for original, prepared in zip(mentions, prepared_mentions):
    expected = dict(original, comparison_name=original["name"].strip())
    assert prepared == expected, "원래 필드를 유지하고 comparison_name만 추가하세요."
assert all("comparison_name" not in row for row in mentions), "원본 출현 기록을 수정하지 마세요."
print("✅ 통과!")


## 4. 같은 이름에서도 타입을 확인합니다

**배경**: API 설명 문서와 그 문서가 설명하는 API 요소는 이름이 같아도 서로 다른 개체입니다. 비교할 쌍에서 제외하더라도 원래 출현 기록은 남깁니다.  

**요구사항**  
- **same_name_pairs** 에 prepared_mentions에서 comparison_name이 같은 두 기록의 `mention_id` 쌍을 집합으로 담으세요. 세 산출물 모두 두 mention_id를 `pair_key`로 정렬한 튜플로 표현합니다.
- **compatible_pairs** 에는 same_name_pairs 중 entity_type도 같은 쌍만 집합으로 담으세요.
- **excluded_pairs** 에 타입이 달라 제외한 쌍을 집합으로 담고 두 기록의 이름과 타입을 출력하세요.

**확인 기준**: 이름이 같은 16쌍 중 타입이 같은 후보는 13쌍, 타입이 달라 제외하는 쌍은 3쌍입니다. 같은 타입이라는 조건도 동일 개체 확정의 충분조건은 아닙니다.  

<details><summary>힌트</summary>

```text
접근방법:
- combinations로 서로 다른 두 출현 기록을 고르고 이름 조건과 타입 조건을 따로 검사합니다.

세부구현:
1. 비교용 이름이 같은 쌍을 먼저 모읍니다.
2. 타입까지 같은 쌍만 compatible_pairs에 넣습니다.
3. 차집합으로 제외된 쌍을 구합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
expected_name_pairs = set()
expected_compatible = set()
for left, right in combinations(prepared_mentions, 2):
    if left["comparison_name"] == right["comparison_name"]:
        key = pair_key(left["mention_id"], right["mention_id"])
        expected_name_pairs.add(key)
        if left["entity_type"] == right["entity_type"]:
            expected_compatible.add(key)
assert same_name_pairs == expected_name_pairs, "쌍은 pair_key로 정렬한 튜플로 담고 이름이 같은 쌍을 빠짐없이 모으세요."
assert compatible_pairs == expected_compatible, "쌍은 pair_key로 정렬한 튜플로 담고 다른 타입의 기록을 섞지 마세요."
assert excluded_pairs == expected_name_pairs - expected_compatible, "제외한 쌍도 원래 출현 ID로 보존하세요."
print("✅ 통과!")


## 5. 반복 비교를 줄일 대표 표기를 고릅니다

**배경**: 같은 타입과 표기를 매번 비교하지 않도록 후보 검색에 사용할 대표 기록을 고릅니다.  

**요구사항**  
- **`representatives`** 에 `(entity_type, comparison_name)` 튜플을 키로, prepared_mentions에서 처음 나온 해당 기록을 값으로 담은 딕셔너리를 만드세요.
- **`representatives`** 를 만들 때 prepared_mentions와 mentions를 수정하거나 출현 기록을 삭제하지 마세요.

**확인 기준**: 각 타입·표기 조합은 한 번만 나오며, 같은 이름의 문서와 함수는 별도 키로 남습니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 새 키가 처음 나왔을 때만 대표 기록으로 등록합니다.

세부구현:
1. 타입과 비교용 이름을 묶어 키를 만듭니다.
2. 키가 아직 없을 때만 기록을 저장합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
# 원래 트리플에서 기대하는 출현을 다시 만들어 원본 변경도 검사합니다.
expected_mentions = []
expected_prepared = []
expected_representatives = {}
for triple in original_triples:
    for role in ["subject", "object"]:
        original = {"mention_id": triple["triple_id"] + ":" + role,
                    "triple_id": triple["triple_id"], "role": role,
                    "name": triple[role], "entity_type": triple[role + "_type"],
                    "source_doc_id": triple["source_doc_id"], "evidence": triple["evidence"]}
        expected_mentions.append(original)
        prepared = dict(original, comparison_name=normalize(original["name"]))
        expected_prepared.append(prepared)
        expected_representatives.setdefault((prepared["entity_type"], prepared["comparison_name"]), prepared)
assert mentions == expected_mentions, "원래 출현 기록의 순서와 모든 필드를 보존하세요."
assert prepared_mentions == expected_prepared, "비교용 이름을 포함한 출현 기록을 변경하지 마세요."
assert representatives == expected_representatives, "타입과 비교용 이름을 묶고 처음 출현한 기록을 선택하세요."
assert len(mentions) == 16 and len(prepared_mentions) == 16, "대표를 골라도 원래 출현은 남겨 두세요."
print("✅ 통과!")


In [ ]:
# [제공코드] 기준값을 바꾸면 결과가 달라지는지 확인할 이름 쌍입니다. 저장 자료와 분리한 검사 입력입니다.
similarity_probe_pairs = [
    ("displot", "histplot"),
    ("kdeplot", "displot"),
    ("violinplot", "countplot"),
    ("regplot", "lmplot"),
    ("boxplot", "heatmap"),
]
pprint(similarity_probe_pairs)


## 6. 문자열 유사도로 검토 후보를 찾습니다

**배경**: 표기가 조금 다른 함수도 원문 검토 대상에 포함합니다.  

**요구사항**  
- **`name_candidates`** 에 representatives의 값들을 입력 순서대로 두 개씩 비교하여, 타입이 같고 comparison_name의 SequenceMatcher 유사도가 0.65 이상인 쌍을 딕셔너리 목록으로 담으세요.
- **`name_candidates`** 의 각 행은 `left_id`, `right_id`, `similarity` 세 키를 가집니다. 두 ID는 비교한 왼쪽·오른쪽 기록의 mention_id이고 similarity는 반올림하지 않은 ratio 값입니다. combinations 순서를 유지하세요.
- **`probe_candidates`** 에 similarity_probe_pairs 중 같은 0.65 기준을 통과하는 쌍만 입력 순서대로 담으세요. 각 원소는 점수를 뺀 `(왼쪽 이름, 오른쪽 이름)` 두 칸 튜플이며, 원래 쌍을 그대로 씁니다. 이 검사 입력에는 타입이 없으므로 이름만 비교합니다.

**확인 기준**: 점수는 0~1 범위이며 동일 개체일 확률이 아닙니다. 조건에 맞는 쌍을 빠짐없이 포함해야 합니다. 검사 입력 5쌍 중에는 기준값을 0.6으로 낮추면 통과하지만 0.65에서는 떨어지는 쌍이 있습니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 같은 타입의 대표 표기만 비교하고, 점수 조건을 통과한 기록을 모읍니다.

세부구현:
1. combinations로 서로 다른 대표 기록 두 개를 고릅니다.
2. 타입이 다르면 제외합니다.
3. SequenceMatcher의 ratio로 점수를 구하고 기준 이상만 남깁니다.
4. 같은 기준값을 검사 입력의 이름 쌍에도 적용합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
expected_candidates = []
for left, right in combinations(representatives.values(), 2):
    score = SequenceMatcher(None, left["comparison_name"], right["comparison_name"]).ratio()
    if left["entity_type"] == right["entity_type"] and score >= 0.65:
        expected_candidates.append({"left_id": left["mention_id"], "right_id": right["mention_id"], "similarity": score})
assert len(name_candidates) == len(expected_candidates), "타입과 0.65 이상 조건을 확인하고 후보를 빠짐없이 모으세요."
for actual, expected in zip(name_candidates, expected_candidates):
    assert set(actual) == set(expected), "후보의 세 키를 확인하세요."
    assert (actual["left_id"], actual["right_id"]) == (expected["left_id"], expected["right_id"]), "대표 기록의 순서와 ID를 확인하세요."
    assert abs(actual["similarity"] - expected["similarity"]) < 1e-12, "점수를 반올림하지 않고 저장하세요."
# 저장 자료는 0.6과 0.65가 같은 결과를 주므로 기준값 자체는 검사 입력으로 확인합니다.
assert probe_candidates == [("displot", "histplot"), ("kdeplot", "displot")], "기준값 0.65를 그대로 적용하세요. 0.6으로 낮추면 두 쌍이 더 들어옵니다."
print("✅ 통과!")


## 7. 원문 문맥으로 동일 개체 여부를 판정합니다(서술형)

**배경**: 문자열 유사도만으로는 별칭과 서로 다른 함수를 구분할 수 없습니다.  

**요구사항**  
- **판정 메모**에 아래 후보마다 두 이름, 같음·다름·보류 판정, 문서 ID와 근거를 서술하세요.
- **판정 메모**에는 타입이 같고 이름이 비슷해도 통합하면 안 되는 이유를 포함하세요.
- **판정 메모**의 마지막에 구조화 출력의 형식이 맞아도 판정이 옳다고 단정할 수 없는 이유를 한 문장으로 적으세요.

**확인 기준**: task_documents의 본문에서 import와 호출을 대조하세요. 후보마다 판정과 근거 문서 ID가 있고, 마지막 한 문장이 형식과 사실성의 차이를 말합니다.  

<details><summary>힌트</summary>

```text
접근방법:
- sns가 어느 모듈인지와 함수 이름 자체가 같은지를 따로 확인합니다.

세부구현:
1. 출현 기록에서 원문 문서를 찾습니다.
2. import와 호출 행을 읽어 같은 함수의 다른 표기인지 확인합니다.
3. 확인할 수 없으면 보류하고 추가로 필요한 정보를 적습니다.
```

</details>


In [ ]:
# [제공코드] 후보 기록의 원문 근거를 확인한 뒤 다음 서술란에서 직접 판정합니다.
by_mention_id = {row["mention_id"]: row for row in mentions}
for candidate in name_candidates:
    for key in ["left_id", "right_id"]:
        mention = by_mention_id[candidate[key]]
        print(mention["mention_id"], mention["name"], mention["entity_type"])
        print(mention["source_doc_id"], mention["evidence"])
    print()


*(여기에 후보별 판정과 원문 근거, 구조화 출력의 한계를 서술하세요. 이 서술란은 자동 채점 대상이 아니므로 정답 노트북의 모범 서술과 직접 대조하세요.)*


7번에서는 검색 후보를 원문으로 직접 판정했습니다.  
8~10번은 같은 자료의 저장된 원문 검토를 사용해 **그룹 만들기 → ID 부여 → 저장**을 이어갑니다.  


### 그룹으로 묶을 판정 기록을 읽습니다

이 파일은 **기존 출현별 원문 검토를 같은 개체 쌍으로 바꾼 자료**입니다.  
검색 후보와 달리 원문 판정이 끝난 쌍이며, 양쪽 판정 이유도 남아 있습니다.  
새 모델 호출이나 골드 조회 없이 이 판정에서 그룹을 만들고 ID를 부여합니다.  


In [ ]:
# [제공코드] kg_lv1_pair_review.json: 이 과제 출현의 원문 검토와 같은 개체 쌍입니다.
pair_review = json.loads((data_dir / "kg_lv1_pair_review.json").read_text(encoding="utf-8"))
pair_decisions = pair_review["decisions"]

# 다른 추출 버전의 판정을 잘못 적용하지 않도록 전체 출현을 대조합니다.
assert pair_review["mentions"] == mentions, "판정 파일의 출현과 과제 입력이 다릅니다."
print("판정 출처:", pair_review["source"])
print("확인된 같은 개체 쌍:", len(pair_decisions))
pprint(pair_decisions[:2])


In [ ]:
# [제공코드] 같다고 확인한 쌍을 그룹으로 모읍니다. 연결되지 않은 기록도 한 개짜리 그룹으로 남깁니다.
def group_pairs(mention_ids, same_pairs):
    """같은 개체로 판정한 쌍을 이어 출현 그룹을 만듭니다.

    Args:
        mention_ids (list[str]): 보존할 전체 출현 ID.
        same_pairs (list[tuple[str, str]]): 같은 개체로 판정한 출현 ID 쌍.

    Returns:
        list[list[str]]: 정렬한 그룹 목록. 연결되지 않은 출현도 단독 그룹으로 남습니다.
    """
    groups = [{mention_id} for mention_id in mention_ids]
    for left_id, right_id in same_pairs:
        # 평가 범위 밖의 ID를 잘못 넣으면 기록이 빠질 수 있으므로 먼저 확인합니다.
        if left_id not in mention_ids or right_id not in mention_ids:
            raise ValueError("동일 판정 쌍에 입력 목록에 없는 ID가 있습니다.")
        joined = set()
        remaining = []
        for group in groups:
            if left_id in group or right_id in group:
                joined.update(group)
            else:
                remaining.append(group)
        remaining.append(joined)
        groups = remaining
    return sorted([sorted(group) for group in groups])

# 입력 예시: 두 초판 표기는 같은 책이고 개정판은 별도 책입니다.
example_ids = ["d01:object", "d02:object", "d03:object"]
example_pairs = [("d01:object", "d02:object")]  # 파이썬 입문과 파이썬 입문서

# 호출: 같은 초판끼리 묶고 개정판(d03:object)은 따로 남깁니다.
print(group_pairs(example_ids, example_pairs))


## 8. 같은 개체로 판정된 출현을 그룹으로 묶습니다

**배경**: 직접 비교한 쌍뿐 아니라 같은 대상으로 이어지는 출현들도 한 그룹으로 모읍니다.  

**요구사항**  
- **same_pairs** 에 pair_decisions 중 decision이 `같음`인 행의 left_id와 right_id를 pair_key로 정렬한 튜플로 담으세요. 리스트로 만들고 판정 입력 순서를 유지합니다.
- **mention_ids** 에 mentions의 mention_id를 입력 순서대로 담으세요.
- **identity_groups** 에 group_pairs로 전체 mention_ids를 same_pairs에 따라 묶은 결과를 담으세요. 다른 출현과 연결되지 않은 기록도 단독 그룹으로 남깁니다.
- **identity_groups** 와 그룹 수를 출력하세요.

**확인 기준**: 전체 16개 출현이 8개 그룹에 정확히 한 번씩 포함됩니다. 같은 이름이나 역할을 기준으로 임의로 추가 통합하지 않습니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 검색 점수 대신 원문 판정이 끝난 쌍을 그룹 함수에 전달합니다.

세부구현:
1. 같음 판정의 두 출현 ID를 쌍으로 모읍니다.
2. 단독 출현까지 포함한 전체 ID 목록을 만듭니다.
3. 그룹 함수를 적용하고 전체 출현의 보존 여부를 확인합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
expected_pairs = []
for row in pair_decisions:
    if row["decision"] == "같음":
        expected_pairs.append(pair_key(row["left_id"], row["right_id"]))
assert same_pairs == expected_pairs, "같음으로 판정한 쌍만 입력 순서대로 담으세요."
assert mention_ids == [row["mention_id"] for row in mentions], "단독 출현도 전체 목록에 포함하세요."
assert identity_groups == group_pairs(mention_ids, expected_pairs), "판정 쌍의 연결과 이행성을 그룹에 반영하세요."
assert len(identity_groups) == 8, "16개 출현을 8개 동일 개체 그룹으로 묶으세요."
all_members = []
for group in identity_groups:
    all_members.extend(group)
assert sorted(all_members) == sorted(mention_ids), "출현이 빠지거나 여러 그룹에 중복되면 안 됩니다."
print("✅ 통과!")


### 확정된 그룹마다 ID 하나를 만듭니다

`assign_group_ids`는 그룹에서 사전순 첫 출현 ID 앞에 `entity:`를 붙입니다.  
예를 들어 두 출현이 한 그룹이면 **같은 ID**, 서로 다른 그룹이면 **다른 ID**를 부여합니다.  
그룹에 포함되지 않은 보류 출현은 `review`, `None`으로 남습니다. ID를 고르기 위한 LLM 호출은 없습니다.  


In [ ]:
# [제공코드] 같은 개체로 확정한 그룹에 ID 하나를 부여하고 원래 출현 기록에 붙입니다.
def assign_group_ids(mentions, groups):
    """같은 그룹에 같은 표준 ID를 붙인 출현 목록을 반환합니다.

    Args:
        mentions (list[dict]): 원래 출현 기록 전체.
        groups (list[list[str]]): 판정 충돌이 없는 그룹의 출현 ID 목록.

    Returns:
        list[dict]: 원본에 standard_id와 status를 추가한 복사본.
            그룹에 없는 출현은 ID 없이 review 상태로 남깁니다.
    """
    by_id = {row["mention_id"]: row for row in mentions}
    if len(by_id) != len(mentions):
        raise ValueError("출현 ID가 중복되었습니다.")

    id_by_mention = {}
    for group in groups:
        if not group or len(group) != len(set(group)):
            raise ValueError("그룹이 비었거나 같은 출현이 중복되었습니다.")
        if not set(group).issubset(by_id):
            raise ValueError("그룹에 원본에 없는 출현 ID가 있습니다.")
        types = {by_id[mention_id]["entity_type"] for mention_id in group}
        if len(types) != 1:
            raise ValueError("타입이 다른 출현의 판정을 다시 확인하세요.")

        # 정렬상 첫 출현 ID를 사용해 같은 그룹에 다시 실행해도 같은 ID를 만듭니다.
        standard_id = "entity:" + min(group)
        for mention_id in group:
            if mention_id in id_by_mention:
                raise ValueError("한 출현이 여러 그룹에 들어 있습니다.")
            id_by_mention[mention_id] = standard_id

    links = []
    for row in mentions:
        standard_id = id_by_mention.get(row["mention_id"])
        status = "linked" if standard_id is not None else "review"
        links.append(dict(row, standard_id=standard_id, status=status))
    return links


# 예시 입력: 두 표기는 같은 초판, 개정판은 별도 그룹입니다.
id_example_mentions = [
    {"mention_id": "d01:object", "name": "파이썬 입문", "entity_type": "Book"},
    {"mention_id": "d02:object", "name": "파이썬 입문서", "entity_type": "Book"},
    {"mention_id": "d03:object", "name": "파이썬 입문(개정판)", "entity_type": "Book"},
]
id_example_groups = [["d01:object", "d02:object"], ["d03:object"]]
for row in assign_group_ids(id_example_mentions, id_example_groups):
    print("이름:", row["name"], "/ 표준 ID:", row["standard_id"])


## 9. 그룹별 표준 ID를 부여하고 상태를 확인합니다

**배경**: 동일 개체 판정이 끝난 그룹에는 새 ID 하나만 만들면 됩니다.  

**요구사항**  
- **entity_links** 에 assign_group_ids로 mentions와 identity_groups를 처리한 전체 기록을 담으세요.
- **status_counts** 에 linked와 review의 건수를 딕셔너리로 담으세요. 없는 상태도 0으로 기록합니다.
- **linked_id_map(links)** 함수를 작성하세요. status가 linked인 행만 `mention_id: standard_id` 딕셔너리로 반환합니다.
- **mention_to_id** 에 entity_links를 linked_id_map으로 처리한 결과를 담고 출력하세요.

**확인 기준**: 16개 출현이 8개 ID에 연결됩니다. 각 기록은 원래 7개 필드와 standard_id, status의 9개 필드를 가집니다. 같은 그룹은 같은 ID를 공유하며 원래 출현 기록은 그대로 남습니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 그룹별 ID를 부여한 결과에서 확정 상태만 골라 ID 조회 사전을 만듭니다.

세부구현:
1. 확정된 그룹과 전체 출현을 ID 부여 함수에 전달합니다.
2. linked와 review를 각각 셉니다.
3. linked 행만 ID 조회 사전에 담는 함수를 만듭니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert entity_links == assign_group_ids(mentions, identity_groups), "각 그룹에 ID 하나를 부여하고 원래 출현을 보존하세요."
assert status_counts == {"linked": 16, "review": 0}, "0건인 review도 포함해 집계하세요."
expected_mapping = {}
for group in identity_groups:
    for mention_id in group:
        expected_mapping[mention_id] = "entity:" + min(group)
assert mention_to_id == expected_mapping, "같은 그룹은 같은 ID를, 다른 그룹은 다른 ID를 가져야 합니다."
mixed_links = [
    {"mention_id": "case:1", "standard_id": "entity:case:1", "status": "linked"},
    {"mention_id": "case:2", "standard_id": None, "status": "review"},
]
assert linked_id_map(mixed_links) == {"case:1": "entity:case:1"}, "ID 부여를 보류한 행은 조회 사전에 넣지 마세요."
assert len(set(mention_to_id.values())) == 8, "표준 ID의 수는 확정한 그룹 수와 같습니다."
print("✅ 통과!")


## 10. 출현별 연결 기록을 파일로 저장합니다

**배경**: 다음 처리 단계에서 읽을 수 있도록 ID 연결과 원래 근거를 함께 저장합니다.  

**요구사항**  
- **`links_path`** 에 `output/lv1_entity_links.jsonl`의 Path를 담고 부모 폴더가 없으면 만드세요.
- **`links_path`** 에 entity_links를 입력 순서대로 한 줄당 JSON 하나로 UTF-8 저장하세요. ensure_ascii는 False로 하고 마지막에도 줄바꿈을 넣으세요.
- **`saved_links`** 에 저장한 파일을 다시 읽어 복원한 딕셔너리 목록을 담으세요.

**확인 기준**: 원래 이름·타입·출처·근거와 status·standard_id를 포함한 16행이 다시 읽힙니다. 교안에서 저장하는 파일과 구분해 과제 전용 파일명을 사용합니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 각 기록을 JSON 문자열로 바꿔 줄바꿈으로 연결하고, 저장한 내용을 다시 파싱합니다.

세부구현:
1. Path로 저장 경로와 부모 폴더를 준비합니다.
2. json.dumps와 줄바꿈으로 JSONL을 저장합니다.
3. 준비 셀의 load_rows는 data 폴더 전용이므로 저장한 파일은 links_path에서 직접 읽습니다.
4. 각 줄을 json.loads로 복원합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert links_path.resolve() == (Path("output") / "lv1_entity_links.jsonl").resolve(), "과제 전용 파일 경로를 사용하세요."
stored_text = links_path.read_text(encoding="utf-8")
assert stored_text.endswith("\n") and len(stored_text.splitlines()) == len(entity_links), "한 줄에 한 기록을 저장하고 마지막 줄바꿈도 넣으세요."
assert [json.loads(line) for line in stored_text.splitlines()] == entity_links, "파일의 각 행에 원래 연결 결과를 보존하세요."
assert saved_links == entity_links, "다시 읽은 값이 원래 연결 결과와 같아야 합니다."
assert task_triples == original_triples, "원래 트리플을 변경하지 마세요."
print("✅ 통과!")


In [ ]:
# [제공코드] 실제 문서와 분리한 작은 이웃 집합으로 교집합과 합집합의 역할을 확인합니다.
neighbor_examples = {
    "partial": ({"doc_a", "doc_b"}, {"doc_b", "doc_c"}),
    "disjoint": ({"doc_a"}, {"doc_c"}),
    "identical": ({"doc_a", "doc_b"}, {"doc_a", "doc_b"}),
    "empty": (set(), set()),
}
pprint(neighbor_examples)


## 11. GDS 후보 검색의 입력과 점수를 구별합니다

**배경**: GDS를 실행하기 전에 두 검색 방법이 무엇을 비교하는지, 높은 점수로 무엇까지 판단할 수 있는지 확인합니다. 이 문항에는 API나 DB가 필요하지 않습니다.  

**요구사항**  
- **`neighbor_jaccard`** 함수를 작성하세요. `left_docs`, `right_docs` 두 문서 ID 집합을 받아 Jaccard 유사도(공통 문서 수 / 두 집합 전체의 서로 다른 문서 수)를 float로 반환합니다. 합집합이 비어 있으면 이 과제에서는 0.0을 반환하고, 입력 집합은 수정하지 않습니다.
- **`neighbor_scores`** 에 neighbor_examples의 이름을 키로, 해당 두 집합을 neighbor_jaccard로 비교한 점수를 값으로 담은 딕셔너리를 만드세요. 값은 반올림하지 않으며 자가채점의 절대 허용오차는 1e-12입니다.
- **비교 메모** 에 이 교안의 KNN과 Node Similarity가 각각 입력으로 삼는 정보가 무엇인지 설명하세요. 또 서로 다른 두 함수가 완전히 같은 문서 집합에 등장하고 두 검색 방법 모두 후보로 선택한 경우, 동일 개체로 확정할 수 있는지와 추가 확인할 근거를 서술하세요. 서술은 정답 노트북의 모범 서술과 비교합니다.

**확인 기준**: neighbor_scores는 neighbor_examples와 같은 네 키와 float 점수를 가지며 모두 0~1 범위입니다. 비교 메모에는 방법별 입력, 높은 점수의 의미, 확정에 필요한 확인을 포함합니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 공통 이웃이 많다는 사실과 같은 개체라는 판정을 나누어 생각합니다.

세부구현:
1. 교집합과 합집합의 크기를 구하고 분모가 비어 있는 경우를 처리합니다.
2. 각 검사 입력의 점수를 이름별로 저장합니다.
3. 교안 01의 두 검색 입력과 원문 판정 단계를 대조해 서술합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
expected_examples = {
    "partial": ({"doc_a", "doc_b"}, {"doc_b", "doc_c"}),
    "disjoint": ({"doc_a"}, {"doc_c"}),
    "identical": ({"doc_a", "doc_b"}, {"doc_a", "doc_b"}),
    "empty": (set(), set()),
}
assert neighbor_examples == expected_examples, "제공된 이웃 집합을 변경하지 마세요."
assert set(neighbor_scores) == set(expected_examples), "네 입력 이름을 빠짐없이 키로 사용하세요."
for key, (left, right) in expected_examples.items():
    expected_score = len(left & right) / len(left | right) if left | right else 0.0
    assert isinstance(neighbor_scores[key], float), "점수는 float로 담으세요."
    assert abs(neighbor_scores[key] - expected_score) < 1e-12, "교집합 크기를 합집합 크기로 나누고 반올림하지 마세요."
    left_copy, right_copy = set(left), set(right)
    actual_score = neighbor_jaccard(left_copy, right_copy)
    assert isinstance(actual_score, float) and abs(actual_score - expected_score) < 1e-12, "함수도 각 입력에서 같은 계산 규칙을 따라야 합니다."
    assert left_copy == left and right_copy == right, "함수의 입력 집합을 변경하지 마세요."
# 양쪽 집합의 크기가 달라도 합집합을 직접 구해야 합니다.
for left, right in [({"doc_a", "doc_b"}, {"doc_b"}), ({"doc_b"}, {"doc_a", "doc_b"})]:
    left_before, right_before = set(left), set(right)
    actual_score = neighbor_jaccard(left, right)
    assert isinstance(actual_score, float) and abs(actual_score - 0.5) < 1e-12, "크기가 다른 집합도 실제 합집합의 크기로 나누세요."
    assert left == left_before and right == right_before, "비교 방향이 바뀌어도 입력 집합을 보존하세요."
print("✅ 통과!")


*(여기에 방법별 비교 입력과 후보를 확정하기 전에 확인할 근거를 서술하세요. 이 서술란은 자동 채점 대상이 아니므로 정답 노트북의 모범 서술과 직접 대조하세요.)*


## GDS 후보 검색을 준비합니다(선택 실습)

교안 01의 GDS(Graph Data Science) 준비 코드를 과제의 대표 표기에 적용합니다.  
바로 아래 셀에서 실습용 Neo4j에 연결하고 GDS 플러그인으로 실행합니다.  
이후 GDS 준비 셀에서는 과제 전용 Lv1EntityCandidate와 Lv1SourceDocument 라벨의 노드만 초기화합니다.  
KNN은 OpenAI로 생성한 이름과 타입의 임베딩을, Node Similarity는 공통 문서 이웃을 비교합니다.  
임베딩 생성에는 .env의 OPENAI_API_KEY가 필요합니다.  
선택 실습을 하지 않으면 필수 11번까지 완료한 뒤 마칩니다.  


In [ ]:
# [제공코드] 노드 초기화와 적재에 사용할 실습 전용 Neo4j에 연결합니다.
import os
from urllib.parse import urlsplit
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase

# 현재 작업 폴더부터 상위로 올라가 가장 가까운 .env를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
neo4j_uri = os.environ["NEO4J_URI"]
# driver는 여러 쿼리에서 재사용할 DB 연결 통로입니다. 계정 정보는 출력하지 않습니다.
driver = GraphDatabase.driver(
    neo4j_uri,
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
)
# 연결 객체 생성만으로 접속 성공이 보장되지 않으므로 지금 서버 접속을 확인합니다.
driver.verify_connectivity()

def run_cypher(query, **params):
    """값을 매개변수로 전달하고 Cypher 결과를 딕셔너리 목록으로 돌려줍니다."""
    # 쿼리마다 세션을 열고 with 블록이 끝나면 닫습니다. driver는 계속 재사용합니다.
    with driver.session() as session:
        # RETURN에서 붙인 별칭이 딕셔너리 키가 되어 파이썬에서 조회할 수 있습니다.
        return [record.data() for record in session.run(query, **params)]

# 주소에 계정 정보가 포함되어 있어도 호스트와 포트만 확인합니다.
connection_address = urlsplit(neo4j_uri)
print("Neo4j 연결 완료. 호스트:", connection_address.hostname, "/ 포트:", connection_address.port)


### A. KNN으로 임베딩 벡터가 가까운 후보를 찾습니다

**비교 대상은 이름과 타입을 변환한 임베딩 벡터입니다.**  
예를 들어 `ApiElement: kdeplot`을 모델에 보내면 그 텍스트를 나타내는 숫자 목록을 받습니다.  
같은 모델로 만든 벡터끼리 코사인 유사도를 비교해 가까운 후보를 찾습니다.  

`OpenAIEmbeddings`는 OpenAI 임베딩 API를 호출하는 LangChain 클래스입니다.  
대화에 쓰는 `gpt-5.6-luna`와 구분해, 임베딩 전용 **text-embedding-3-large**을 사용합니다.  
`embed_documents(문자열 목록)`은 입력 순서대로 벡터 목록을 반환합니다.  

- **입력:** 대표 기록마다 `타입: 비교용 이름` 형태로 만든 문자열
- **모델 출력:** 입력 하나당 768차원의 벡터 하나 (`dimensions=768`로 지정)
- **KNN 결과:** 벡터가 가까워 원문으로 확인할 후보 쌍

이름과 타입만 임베딩합니다. 원문 근거나 정답 ID를 임베딩 입력에 넣지 않습니다.  
이름의 의미가 비슷해도 서로 다른 함수일 수 있으므로 높은 점수만으로 ID를 합치지 않습니다.  

[OpenAI 임베딩 모델](https://developers.openai.com/api/docs/models/text-embedding-3-large),  
[OpenAIEmbeddings 사용법](https://docs.langchain.com/oss/python/integrations/embeddings/openai)  


In [ ]:
# [제공코드] 이름과 타입을 OpenAI 임베딩 모델에 보내 비교할 숫자 벡터를 만듭니다.
from langchain_openai import OpenAIEmbeddings

# 입력과 응답의 순서를 맞추려고 대표 기록을 리스트로 고정합니다.
representative_rows = list(representatives.values())
embedding_texts = []
for row in representative_rows:
    embedding_texts.append(f'{row["entity_type"]}: {row["comparison_name"]}')

# 대화 모델과 달리 이 모델은 텍스트마다 숫자 벡터를 반환합니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    check_embedding_ctx_length=False,  # LangChain의 길이 검사와 자동 분할 없이 문자열을 그대로 보냅니다.
    # API가 처음부터 768차원 벡터를 반환하도록 요청합니다. 반환 후 잘라내지 않습니다.
    dimensions=768,
)

# .env의 OPENAI_API_KEY로 실제 API를 호출합니다. 다시 실행하면 다시 요청합니다.
vectors = embedding_model.embed_documents(embedding_texts)

# zip은 같은 위치의 기록과 벡터를 짝짓습니다. strict=True는 개수가 다르면 오류를 냅니다.
candidate_nodes = []
for row, vector in zip(representative_rows, vectors, strict=True):
    candidate_nodes.append(dict(row, vector=vector))

print("임베딩 입력 수:", len(embedding_texts))
print("벡터 수:", len(vectors), "/ 벡터 차원:", len(vectors[0]))
print("첫 입력:", embedding_texts[0])
print("첫 벡터의 앞 5개 값:", vectors[0][:5])


**임베딩을 노드 속성에 저장하고 KNN용 메모리 그래프를 만듭니다.**  

- **DB 노드:** `Lv1EntityCandidate`에 대표 기록과 임베딩 `vector`를 저장합니다.
- **메모리 그래프:** `lv1_name_vectors`에 위 노드와 벡터를 가져옵니다.
- **확인:** 출력에서 투영된 대표 노드 수를 확인하세요.


In [ ]:
# [제공코드] GDS는 후보 검색용 그래프를 메모리에 투영해 계산합니다. 최종 지식 그래프 적재는 교안 02에서 합니다.
# 재실행해도 후보가 누적되지 않도록 이 실습의 검색용 노드만 비웁니다.
# DETACH DELETE는 선택한 노드와 그 노드에 연결된 관계를 함께 삭제합니다.
run_cypher("MATCH (n:Lv1EntityCandidate) DETACH DELETE n")

# 같은 타입과 표기의 대표 기록마다 검색용 노드 하나를 만듭니다.
for row in candidate_nodes:
    run_cypher("""
    // 같은 대표 출현 ID의 노드가 있으면 재사용하고, 없으면 만듭니다.
    MERGE (n:Lv1EntityCandidate {mention_id: $mention_id})
    // vector는 OpenAI가 이름과 타입에서 만든 임베딩입니다. KNN이 이 값을 비교합니다.
    SET n.name = $name, n.entity_type = $entity_type, n.vector = $vector
    """, mention_id=row["mention_id"], name=row["name"],
        entity_type=row["entity_type"], vector=row["vector"])
# 이전 실행의 메모리 그래프만 해제합니다. DB에 저장한 노드는 삭제하지 않습니다.
# false는 같은 이름의 메모리 그래프가 없어도 오류를 내지 않도록 합니다.
run_cypher("CALL gds.graph.drop('lv1_name_vectors', false) YIELD graphName")

# 투영은 DB의 필요한 노드와 속성을 GDS 계산용 메모리 그래프로 가져오는 작업입니다.
run_cypher("""
// 이름 비교에는 후보 노드와 vector 속성을 가져옵니다.
// '*'는 선택한 후보 노드 사이의 모든 관계 유형입니다. KNN은 관계 없이도 벡터로 계산합니다.
CALL gds.graph.project('lv1_name_vectors',
    {Lv1EntityCandidate: {properties: ['vector']}}, '*')
YIELD graphName
""")


**KNN으로 가까운 후보를 구합니다.**  

| 설정 | 의미 |
|---|---|
| `vector: COSINE` | vector 속성끼리 코사인 유사도를 비교 |
| `topK: 3` | 각 노드에서 가까운 후보를 최대 3개 선택 |
| `similarityCutoff: 0.8` | 이번 검색에서는 GDS 점수 0.8 이상만 선택 |

- **점수:** GDS 점수 0.8은 코사인 유사도 0.6입니다. 문자열 유사도나 정답 확률과 다릅니다.
- **타입:** 최대 3개를 찾은 뒤 다른 타입을 제외합니다. 제외한 자리는 다시 채우지 않습니다.
- **한계:** 근사 탐색이므로 가까운 후보를 놓칠 수 있습니다.


In [ ]:
# [제공코드] COSINE 결과는 GDS에서 (1 + 코사인) / 2로 변환됩니다. 0.8은 코사인 0.6에 해당합니다.
knn_rows = run_cypher("""
// 각 노드의 vector와 가까운 후보를 최대 3개 찾고, 결과 행으로 돌려줍니다.
CALL gds.knn.stream('lv1_name_vectors', {
    nodeProperties: [{vector: 'COSINE'}], topK: 3,
    // 0.8 미만은 제외하고, 시드와 실행 스레드를 고정해 결과를 재현합니다.
    similarityCutoff: 0.8, randomSeed: 42, concurrency: 1
}) YIELD node1, node2, similarity
// 반환된 노드 번호를 실제 노드로 바꿔 이름과 타입 속성에 접근합니다.
WITH gds.util.asNode(node1) AS a, gds.util.asNode(node2) AS b, similarity
// 두 후보의 entity_type이 다르면 검토 후보에서 제외합니다.
WHERE a.entity_type = b.entity_type
// Python에서 원문을 다시 찾도록 두 출현 ID와 점수를 반환합니다.
RETURN a.mention_id AS left_id, b.mention_id AS right_id, similarity
""")
# A-B와 B-A를 같은 쌍으로 바꾸고 중복 후보를 한 번만 남깁니다.
knn_pairs = set()
for row in knn_rows:
    knn_pairs.add(pair_key(row["left_id"], row["right_id"]))
print("KNN 후보 쌍 수:", len(knn_pairs))
for row in knn_rows:
    left = by_mention_id[row["left_id"]]
    right = by_mention_id[row["right_id"]]
    print("타입:", left["entity_type"])
    print("이름 쌍:", left["name"], "↔", right["name"])
    print(f'GDS 유사도: {row["similarity"]:.3f}')
    print()


### B. Node Similarity로 공통 출처 문서가 많은 후보를 찾습니다

**이번에는 벡터가 아니라 각 이름이 연결된 문서 집합을 비교합니다.**  
이름이 등장한 모든 문서를 연결한 뒤, 두 이름이 얼마나 많은 출처 문서를 공유하는지 계산합니다.  
임베딩 API를 다시 호출하지 않으며 `vector` 속성도 사용하지 않습니다.  

**Jaccard 유사도 = 공통 문서 수 ÷ 두 이름의 문서 합집합 크기**  
A의 문서가 `{문서1, 문서2}`, B의 문서가 `{문서2, 문서3}`이면 `1 / 3`입니다.  
같은 문서에 나온 서로 다른 개체도 점수가 높을 수 있습니다.  


In [ ]:
# [제공코드] Node Similarity는 공통 출처 문서의 비율을 비교하며 이름 벡터는 사용하지 않습니다.
# 공통 문서를 비교할 출처 노드와 연결을 준비합니다.
run_cypher("MATCH (n:Lv1SourceDocument) DETACH DELETE n")

# 전체 출현을 돌아야 같은 표기가 등장한 모든 문서와 연결할 수 있습니다.
for row in prepared_mentions:
    key = (row["entity_type"], row["comparison_name"])
    # 이 표기를 대표하는 검색용 노드를 찾습니다. 출처는 현재 출현의 문서를 사용합니다.
    representative_id = representatives[key]["mention_id"]
    run_cypher("""
    // 앞에서 만든 대표 노드를 찾고 출처 문서 노드를 준비합니다.
    MATCH (n:Lv1EntityCandidate {mention_id: $mention_id})
    MERGE (d:Lv1SourceDocument {doc_id: $doc_id})
    // 같은 표기와 문서의 연결은 한 번만 만듭니다. 공통 문서 비교에 사용합니다.
    MERGE (n)-[:LV1_APPEARS_IN]->(d)
    """, mention_id=representative_id, doc_id=row["source_doc_id"])

run_cypher("CALL gds.graph.drop('lv1_document_neighbors', false) YIELD graphName")
run_cypher("""
// 공통 이웃을 비교하려면 후보와 문서 노드, 후보에서 문서로 향하는 연결이 모두 필요합니다.
CALL gds.graph.project('lv1_document_neighbors',
    ['Lv1EntityCandidate', 'Lv1SourceDocument'], 'LV1_APPEARS_IN')
YIELD graphName
""")
neighbor_rows = run_cypher("""
// JACCARD는 공통 문서 수를 두 후보의 전체 문서 수(중복 제외)로 나눈 값입니다.
CALL gds.nodeSimilarity.stream('lv1_document_neighbors', {
    // 공통 문서 비율이 절반 이상인 쌍을 찾습니다. top_k는 비교 가능한 다른 후보 수입니다.
    similarityMetric: 'JACCARD', similarityCutoff: 0.5, topK: $top_k
}) YIELD node1, node2, similarity
// 반환된 노드 번호를 실제 노드로 바꿔 이름과 타입 속성에 접근합니다.
WITH gds.util.asNode(node1) AS a, gds.util.asNode(node2) AS b, similarity
// 두 후보의 entity_type이 다르면 검토 후보에서 제외합니다.
WHERE a.entity_type = b.entity_type
// Python에서 원문을 다시 찾도록 두 출현 ID와 점수를 반환합니다.
RETURN a.mention_id AS left_id, b.mention_id AS right_id, similarity
""", top_k=len(candidate_nodes) - 1)
# 두 방향으로 반환된 같은 쌍을 하나로 합칩니다.
neighbor_pairs = set()
for row in neighbor_rows:
    neighbor_pairs.add(pair_key(row["left_id"], row["right_id"]))
print("공통 문서 후보 쌍 수:", len(neighbor_pairs))
for row in neighbor_rows:
    left = by_mention_id[row["left_id"]]
    right = by_mention_id[row["right_id"]]
    print("타입:", left["entity_type"])
    print("이름 쌍:", left["name"], "↔", right["name"])
    print(f'Jaccard 유사도: {row["similarity"]:.3f}')
    print()

# 후보 검색에 사용한 메모리 그래프만 해제합니다. DB의 원래 노드는 남깁니다.
run_cypher("CALL gds.graph.drop('lv1_name_vectors', false) YIELD graphName")
run_cypher("CALL gds.graph.drop('lv1_document_neighbors', false) YIELD graphName")


## 12. 서로 다른 GDS 후보를 합칩니다(선택 실습)

**배경**: 이름 임베딩과 공통 문서에서 찾은 후보를 중복 없이 원문 검토 대상으로 모읍니다.  

**요구사항**  
- **`combined_candidates`** 에 knn_pairs와 neighbor_pairs의 합집합을 집합(set)으로 담으세요.
- **`shared_candidates`** 에 두 집합의 교집합을 집합(set)으로 담으세요.
- **`candidate_evidence`** 는 딕셔너리 리스트로 만드세요. sorted(combined_candidates)의 각 쌍에서 첫 ID를 left_id, 둘째 ID를 right_id로 쓰고, `left_evidence`, `right_evidence`를 포함한 네 키만 담으세요. 후보가 없으면 빈 리스트 []입니다. 근거는 by_mention_id에서 찾아 그대로 복사합니다.

**확인 기준**: 후보 개수는 GDS 실행 결과에 따라 달라질 수 있습니다. 두 방법이 모두 선택했어도 동일 개체로 확정하지 않고 양쪽 원문 근거를 대조합니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 후보 집합을 합치되 각 출현 ID로 원래 근거를 다시 찾습니다.

세부구현:
1. 합집합과 교집합을 따로 계산합니다.
2. 합친 후보를 정렬해 각 출현의 evidence를 조회합니다.
3. 임베딩과 공통 문서는 후보 검색의 근거이며 확정 판정이 아님을 확인합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert combined_candidates == knn_pairs | neighbor_pairs, "두 후보 집합을 빠짐없이 합치세요."
assert shared_candidates == knn_pairs & neighbor_pairs, "두 방법에 공통으로 포함된 후보만 고르세요."
expected_evidence = [{"left_id": a, "right_id": b, "left_evidence": by_mention_id[a]["evidence"], "right_evidence": by_mention_id[b]["evidence"]} for a, b in sorted(combined_candidates)]
assert candidate_evidence == expected_evidence, "두 출현의 원래 근거와 정렬 순서를 확인하세요."
print("✅ 통과!")
